[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/myuyen0304/retail-sales-forecasting-research/blob/main/notebooks/colab_run.ipynb)

# Chạy cross-validation trên Colab

Notebook này chạy mọi thứ trên Colab, laptop không chạy gì:
- mục 4: CV model thống kê và cây (tham số lấy từ `results/best_params.json` đã tune trước);
- mục 5–6: tune RNN/LSTM, rồi CV bản chính (5 seed) và bản `_ar` (3 seed);
- mục 8: dựng bảng và hình cho paper.

Mọi kết quả ghi vào Drive `MyDrive/rsf/results`. Xong thì tải thư mục đó về đặt vào repo (mục 9). Colab bản miễn phí chỉ có 2 lõi nên có thể cần vài phiên.

**Trước khi chạy:**
1. Trên laptop, `git push origin main` để Colab lấy được code mới nhất.
2. Chuẩn bị token Databricks của tài khoản `uyenuyen9780@gmail.com` (Databricks → Settings → Developer → Access tokens → Generate new token). Data được trích thẳng từ DWH, bản `ccde5a5e…`, ở mục 2; tài khoản này đã có quyền đọc schema marts của bản đó.
3. Runtime → Change runtime type → **CPU**. GPU không giúp gì, vì code chạy torch trên CPU.

**Khi đang chạy:**
- Giữ tab mở. Colab bản miễn phí ngắt khi tab bỏ không lâu, và mỗi phiên tối đa khoảng 12 giờ.
- Bị ngắt thì chạy lại từ cell đầu. Mỗi task ghi một file vào Drive, nên task đã xong sẽ được bỏ qua.
- Stage `test` (mục 10) chỉ chạy một lần, sau khi đã chốt phương pháp đề xuất trên CV.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/rsf'
os.makedirs(f'{BASE}/results', exist_ok=True)
os.makedirs(f'{BASE}/data', exist_ok=True)
os.environ['RSF_PROCESSED_DIR'] = f'{BASE}/data'   # data trích từ DWH ở mục 2
os.environ['RSF_RESULTS_DIR'] = f'{BASE}/results'
os.environ['RSF_TORCH_THREADS'] = '1'   # Colab có 2 lõi, chạy --jobs 2 nên mỗi tiến trình 1 luồng
os.environ['PYTHONUTF8'] = '1'
print('CPU:', os.cpu_count())

Mounted at /content/drive
CPU: 2


## 1. Lấy code

Repo private thì nhập token GitHub khi được hỏi. Token chỉ nhập qua ô `getpass`, đừng dán vào cell. Clone xong, token được xóa khỏi cấu hình git.

In [2]:
import getpass, subprocess
REPO = 'github.com/myuyen0304/retail-sales-forecasting-research.git'
if not os.path.exists('/content/rsf'):
    token = getpass.getpass('GitHub token (để trống nếu repo public): ')
    url = f'https://{token}@{REPO}' if token else f'https://{REPO}'
    subprocess.run(['git', 'clone', '--depth', '1', '-q', url, '/content/rsf'], check=True)
    subprocess.run(['git', '-C', '/content/rsf', 'remote', 'set-url', 'origin', f'https://{REPO}'], check=True)
    del token, url
%cd /content/rsf
!git log -1 --oneline

GitHub token (để trống nếu repo public): ··········
/content/rsf
32efdc1 (grafted, HEAD -> main, origin/main, origin/HEAD) Kiem token Databricks bat dau bang dapi truoc khi trich


## 2. Cài thư viện, lấy data từ DWH, kiểm code

Colab có sẵn numpy, pandas và torch, nhưng phiên bản khác laptop. Code mới chỉ được kiểm trên laptop (pandas 3.1).

In [3]:
!pip -q install -e . statsmodels lightgbm xgboost prophet pyarrow -r requirements-dwh.txt
import importlib.metadata as md
print({p: md.version(p) for p in ['pandas', 'numpy', 'joblib', 'torch', 'scikit-learn', 'databricks-sdk']})

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for rsf (pyproject.toml) ... done
{'pandas': '2.2.3', 'numpy': '2.1.3', 'joblib': '1.6.0', 'torch': '2.11.0+cpu', 'scikit-learn': '1.6.1', 'databricks-sdk': '0.117.0'}


Cell dưới trích chuỗi TOTAL + 3 vùng và bảng khuyến mãi từ DWH Databricks (bản `ccde5a5e…`), ghi vào Drive `MyDrive/rsf/data/`, rồi đối chiếu TOTAL với `rpt_forecast_actual` của cùng bản. Script trả lỗi nếu lệch.

- Chỉ chạy một lần: đã có file trên Drive thì bỏ qua, phiên sau không hỏi token nữa.
- Token chỉ nhập qua ô `getpass`. Nó được ghi tạm vào cấu hình Databricks trên máy Colab rồi xóa ngay sau khi trích.
- Đối chiếu từng chuỗi với bản CSV Datathon đã làm trên laptop ngày 2026-10-25: lệch tối đa 1,3e-8, khuyến mãi trùng khớp.

In [4]:
import json
DATA = os.environ['RSF_PROCESSED_DIR']
RELEASE = 'ccde5a5e21c54e9e9054066f'
if not os.path.exists(f'{DATA}/daily_series.csv'):
    cfg = os.path.expanduser('~/.databrickscfg')
    tok = getpass.getpass('Databricks token (dapi...): ').strip()
    if not tok.startswith('dapi'):   # token chỉ hiện một lần lúc tạo; ID trong danh sách token không dùng được
        raise ValueError('Token phải bắt đầu bằng dapi: tạo token mới và chép ngay lúc Databricks hiện nó')
    with open(cfg, 'w') as f:
        f.write(f'[colab]\nhost = https://dbc-ba612f22-93b5.cloud.databricks.com\ntoken = {tok}\n')
    del tok
    try:
        r = subprocess.run(['python', 'scripts/extract_dwh.py', '--profile', 'colab', '--release', RELEASE,
                            '--warehouse-id', '95d9f64890e29b5f', '--out', DATA], capture_output=True, text=True)
    finally:
        os.remove(cfg)
    print(r.stdout, r.stderr[-2000:])
    if r.returncode != 0:
        for f in ('daily_series.csv', 'promotions.csv'):   # xóa file dở để lần sau trích lại
            if os.path.exists(f'{DATA}/{f}'):
                os.remove(f'{DATA}/{f}')
        raise RuntimeError('Trích DWH lỗi hoặc lệch: gửi thông báo ở trên về để sửa')
m = json.load(open(f'{DATA}/manifest.json'))
print(m['release'], m['rows'], 'dòng,', m['promotions'], 'promotions, trích lúc', m['extracted_at'])
print(m['checks'])

ccde5a5e21c54e9e9054066f 15332 dòng, 50 promotions, trích lúc 2026-10-27T16:27:38+00:00
{'total_vs_rpt_forecast_actual': {'unmatched_rows': 0, 'max_abs_gap': 0.00499926891643554}}


Nếu pytest báo lỗi, **dừng lại** và gửi toàn bộ thông báo lỗi về để sửa, đừng chạy tiếp.

In [5]:
!python -m pytest -q

.......                                                                  [100%]
7 passed in 17.10s


## 3. Chép tham số đã tune từ repo sang Drive

Tham số của 7 model thống kê và cây đã tune trên laptop và nằm trong `results/` của repo. Bước này chép chúng sang Drive. Không chép thì lệnh tune sẽ tìm không thấy và có thể chạy lại từ đầu.

In [6]:
import json, shutil
R = os.environ['RSF_RESULTS_DIR']
os.makedirs(f'{R}/tuning', exist_ok=True)
for f in os.listdir('results/tuning'):
    if not os.path.exists(f'{R}/tuning/{f}'):
        shutil.copy(f'results/tuning/{f}', f'{R}/tuning/{f}')
repo_bp = json.load(open('results/best_params.json'))
drive_bp = json.load(open(f'{R}/best_params.json')) if os.path.exists(f'{R}/best_params.json') else {}
json.dump({**repo_bp, **drive_bp}, open(f'{R}/best_params.json', 'w'), indent=2)
print(sorted(json.load(open(f'{R}/best_params.json'))))

['gb', 'lightgbm', 'lstm', 'prophet', 'rf', 'rnn', 'sarima', 'sarimax', 'xgboost']


## 4. CV model thống kê và cây (4 chuỗi × 2 target × 4 fold)

Dòng thứ hai là bản "pure" và SARIMA s=7, dùng cho ablation RQ2.

In [7]:
!python -m rsf.run cv --models baselines,arima,sarima,sarimax,prophet,rf,gb,xgboost,lightgbm,ablation --jobs 2
!python -m rsf.run cv --models sarima_w7,prophet_pure,rf_lag,gb_lag,xgboost_lag,lightgbm_lag --jobs 2

0 task cần chạy
0 task cần chạy


## 5. Tune RNN/LSTM (TOTAL / revenue, 4 fold, 9 cấu hình mỗi model)

Phải chạy xong bước này rồi mới chạy mục 6. Nếu CV chạy trước, kết quả RNN/LSTM bị khóa ở tham số mặc định.

In [8]:
!python -m rsf.run tune --models rnn,lstm --jobs 2
!cat $RSF_RESULTS_DIR/best_params.json

rnn: chọn cấu hình 5 = {'hidden': 32, 'lookback': 56}
lstm: chọn cấu hình 8 = {'hidden': 64, 'lookback': 56}
{
  "sarima": {
    "order": [
      2,
      0,
      1
    ]
  },
  "sarimax": {
    "order": [
      2,
      0,
      2
    ]
  },
  "prophet": {
    "growth": "flat",
    "changepoint_prior_scale": 0.01
  },
  "rf": {
    "min_samples_leaf": 3,
    "max_features": 1.0
  },
  "gb": {
    "learning_rate": 0.03,
    "max_leaf_nodes": 7
  },
  "xgboost": {
    "learning_rate": 0.03,
    "max_depth": 3
  },
  "lightgbm": {
    "learning_rate": 0.03,
    "num_leaves": 7,
    "min_child_samples": 30
  },
  "rnn": {
    "hidden": 32,
    "lookback": 56
  },
  "lstm": {
    "hidden": 64,
    "lookback": 56
  }
}

## 6. CV RNN/LSTM (4 chuỗi × 2 target × 4 fold)

- RNN/LSTM bản chính: 5 seed, 160 task mỗi model.
- Bản `_ar`: 3 seed, 96 task mỗi model.

Bước này lâu nhất. Mục 7 bên dưới cho biết đã chạy được bao nhiêu task và ước tính thời gian còn lại.

In [9]:
_bp = json.load(open(f"{os.environ['RSF_RESULTS_DIR']}/best_params.json"))
assert 'rnn' in _bp and 'lstm' in _bp, 'Chưa tune xong RNN/LSTM: chạy lại mục 5 trước'
!python -m rsf.run cv --models rnn,lstm --jobs 2
!python -m rsf.run cv --models rnn_ar,lstm_ar --jobs 2 --seeds 3

0 task cần chạy
115 task cần chạy
cv rnn_ar WEST revenue F3 0 1305681.0395049937
cv rnn_ar WEST revenue F2 2 894780.402469972
cv rnn_ar WEST revenue F3 1 1024203.3992152276
cv rnn_ar WEST revenue F3 2 560501.9987022758
cv rnn_ar WEST revenue F4 0 391841.48516957025
cv rnn_ar WEST revenue F4 1 1215150.04856971
cv rnn_ar WEST cogs F1 0 760302.8486905642
cv rnn_ar WEST cogs F1 1 466565.1294736454
cv rnn_ar WEST revenue F4 2 389027.46303218394
cv rnn_ar WEST cogs F1 2 441347.1805534543
cv rnn_ar WEST cogs F2 1 1162605.4941422099
cv rnn_ar WEST cogs F2 0 951804.3834098346
cv rnn_ar WEST cogs F2 2 1039815.3495822002
cv rnn_ar WEST cogs F3 0 247681.4554677106
cv rnn_ar WEST cogs F3 1 891760.5578133077
cv rnn_ar WEST cogs F3 2 557259.8667650382
cv rnn_ar WEST cogs F4 0 254132.9778455117
cv rnn_ar WEST cogs F4 1 782494.2703858303
cv rnn_ar WEST cogs F4 2 794022.2725831704
cv lstm_ar TOTAL revenue F1 0 4528725.141658836
cv lstm_ar TOTAL revenue F1 1 3916434.662936709
cv lstm_ar TOTAL revenue F1 

## 7. Theo dõi tiến độ

Trong lúc một cell đang chạy thì không chạy được cell khác. Chạy cell này khi cell kia đã dừng hoặc bị ngắt. Nếu mở phiên mới, chạy lại các cell ở đầu notebook và mục 1–2 trước.

Model thống kê và cây cần 32 file mỗi model. Ước tính giờ còn lại chỉ tính cho RNN/LSTM.

In [10]:
import pandas as pd
from pathlib import Path
R = Path(os.environ['RSF_RESULTS_DIR'])
P = R / 'preds/cv'
count = lambda m: len([p for p in (P / m).glob('*.parquet') if not p.stem.endswith('_fitted')])
print('Đã xong:', {d.name: count(d.name) for d in sorted(P.iterdir())} if P.exists() else {})
need = {'rnn': 160, 'lstm': 160, 'rnn_ar': 96, 'lstm_ar': 96}
done = {m: count(m) for m in need}
print('RNN/LSTM:', {m: f'{done[m]}/{need[m]}' for m in need})
if (R / 'runs_cv.csv').exists():
    log = pd.read_csv(R / 'runs_cv.csv')
    ok = log[log.status == 'ok']
    sec = ok[ok.model.isin(list(need))].groupby('model').fit_s.median()
    print('Số giây fit (trung vị):', sec.round(0).to_dict())
    left = sum((need[m] - done[m]) * sec.get(m, sec.max() if len(sec) else 0) for m in need) / 2 / 3600
    print(f'Ước tính còn khoảng {left:.1f} giờ (với --jobs 2)')
    print('Task lỗi:', (log.status != 'ok').sum())

Đã xong: {'arima': 32, 'gb': 32, 'gb_lag': 32, 'lightgbm': 32, 'lightgbm_lag': 32, 'lightgbm_nodom': 32, 'lightgbm_nopromo': 32, 'lightgbm_noregime': 32, 'lstm': 160, 'lstm_ar': 96, 'prophet': 32, 'prophet_pure': 32, 'rf': 32, 'rf_lag': 32, 'rnn': 160, 'rnn_ar': 96, 'sarima': 32, 'sarima_w7': 32, 'sarimax': 32, 'sarimax_nodom': 32, 'sarimax_nopromo': 32, 'sarimax_noregime': 32, 'snaive1y': 32, 'snaive2y': 32, 'xgboost': 32, 'xgboost_lag': 32}
RNN/LSTM: {'rnn': '160/160', 'lstm': '160/160', 'rnn_ar': '96/96', 'lstm_ar': '96/96'}
Số giây fit (trung vị): {'lstm': 88.0, 'lstm_ar': 102.0, 'rnn': 35.0, 'rnn_ar': 84.0}
Ước tính còn khoảng 0.0 giờ (với --jobs 2)
Task lỗi: 0


## 8. Dựng bảng và hình

Chạy khi mục 7 báo đủ: mỗi model thống kê/cây 32 file, RNN/LSTM đủ số cần. Bảng ghi vào `results/tables`, hình vào `results/figures` trên Drive.

In [11]:
!python scripts/make_tables.py
!python scripts/make_figures.py
print(sorted(os.listdir(f"{os.environ['RSF_RESULTS_DIR']}/tables")))

/content/rsf/src/rsf/results.py:45: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return (preds.groupby(by)
Đã ghi ['T1_main.csv', 'T2_ablation.csv', 'T2b_feature_groups.csv', 'T2b_feature_groups_by_fold.csv', 'T3_ranks.csv', 'T3b_ranks_F2F4.csv', 'T4_bottomup.csv', 'T5_cogs_ratio.csv', 'T6_lead_cv.csv', 'T6_odd_august_cv.csv', 'T8_runtime.csv', 'ensemble_weights.csv', 'scores_cv_by_fold.csv']
/content/rsf/scripts/make_figures.py:181: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  maes = p.groupby("model").ap

## 9. Đặt kết quả vào repo trên laptop (chỉ chép file, không chạy gì)

1. Trên Google Drive (web), chuột phải thư mục `MyDrive/rsf/results` → Download. Drive nén thành file zip, có thể chia nhiều file zip.
2. Cắt thư mục `D:\retail-sales-forecasting-research\results` cũ ra ngoài repo, ví dụ thành `D:\rsf_results_cu` (giữ làm bản sao, đừng xóa).
3. Giải nén file zip, đặt thư mục `results` vào `D:\retail-sales-forecasting-research\results`. Nhiều file zip thì giải nén hết vào cùng thư mục đó.
4. Báo Claude đọc `results/tables`.

## 10. Test: chạy MỘT lần

- Phương pháp đề xuất đã chốt trên CV **trước** khi chạy test: **RNN** (trung bình 5 seed). Người dùng cho phép chạy test ngày 2026-10-28.
- Mọi model fit trên dữ liệu đến 2021-06-30, rồi dự báo 2021-07-01 → 2022-12-31 (549 ngày). Dự báo ghi vào `results/preds/test` trên Drive.
- Mở phiên mới thì chạy trước: cell đầu (mount Drive), mục 1, cell `pip` của mục 2 và mục 3. Data đã có trên Drive nên không hỏi token Databricks.
- Ước tính 2–3 giờ, lâu nhất là RNN/LSTM.
- Bị ngắt thì chạy lại cell. Task đã xong được bỏ qua, nên việc này chỉ làm nốt phần còn thiếu, **không** tính là chạy test lần hai.


In [ ]:
!python -m rsf.run test --models baselines,arima,sarima,sarimax,prophet,rf,gb,xgboost,lightgbm,ablation,sarima_w7,prophet_pure,rf_lag,gb_lag,xgboost_lag,lightgbm_lag --jobs 2
!python -m rsf.run test --models rnn,lstm --jobs 2
!python -m rsf.run test --models rnn_ar,lstm_ar --jobs 2 --seeds 3


Cell dưới kiểm số file test: mỗi model 8 file (4 chuỗi × 2 target); RNN/LSTM 40 (5 seed); bản `_ar` 24 (3 seed). Đủ hết thì in `ĐỦ`.

In [ ]:
import json, subprocess
import pandas as pd
from pathlib import Path
# Lấy danh sách model qua tiến trình python mới, như các lệnh !python ở trên.
# Không import rsf trong kernel: kernel thấy thư mục repo /content/rsf là "gói rsf" rỗng nên import hỏng.
CMD = 'import json; from rsf.models import BASELINES, MAIN, PURE, FEATURE_ABLATION as F; print(json.dumps(BASELINES + MAIN + PURE + F))'
names = json.loads(subprocess.run(['python', '-c', CMD], capture_output=True, text=True, check=True).stdout)
R = Path(os.environ['RSF_RESULTS_DIR'])
P = R / 'preds/test'
count = lambda m: len([p for p in (P / m).glob('*.parquet') if not p.stem.endswith('_fitted')])
need = {m: 8 for m in names}
need.update(rnn=40, lstm=40, rnn_ar=24, lstm_ar=24)
missing = {m: f'{count(m)}/{n}' for m, n in need.items() if count(m) < n}
print('ĐỦ' if not missing else f'Còn thiếu: {missing}')
if (R / 'runs_test.csv').exists():
    log = pd.read_csv(R / 'runs_test.csv')
    print('Task lỗi:', (log.status != 'ok').sum())


Khi cell trên in `ĐỦ`, dựng bảng và hình có thêm cột test (`scores_test.csv`, `T7_dm_test.csv`, hình F08, F10, F11).

Xong thì tải lại `MyDrive/rsf/results` như mục 9, rồi báo Claude.

In [ ]:
!python scripts/make_tables.py --test
!python scripts/make_figures.py --test
print(sorted(os.listdir(f"{os.environ['RSF_RESULTS_DIR']}/tables")))
